In [1]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
from diffusers import DDPMScheduler
import torch
from torch import nn
import torch.nn.functional as F

In [2]:
# Initialize tokenizer and frozen text encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text_encoder = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")
text_encoder.eval()  # Freeze encoder for inference

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [3]:
# Diffusion noise scheduler with 1000 timesteps
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

In [4]:
def text_to_embedding(texts):
    """Convert list of texts to BERT embeddings (last hidden layer)."""
    tokens = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = text_encoder(**tokens, output_hidden_states=True)
    return outputs.hidden_states[-1]  # Shape: [batch_size, seq_len, hidden_dim]

def add_noise(embedding, timesteps):
    """Add noise to embeddings according to the DDPM forward process."""
    noise = torch.randn_like(embedding)
    noisy_embed = noise_scheduler.add_noise(embedding, noise, timesteps)
    return noisy_embed, noise

In [5]:
class SimpleDenoiser(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.linear = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, t=None, cond_emb=None):
      return self.linear(x)

In [6]:
def train_denoiser(texts, epochs=5, batch_size=2, lr=1e-4):
    """Train the denoiser to predict noise given noisy embeddings."""
    denoiser = SimpleDenoiser(hidden_dim=768)
    optimizer = torch.optim.AdamW(denoiser.parameters(), lr=lr)

    for epoch in range(epochs):
        epoch_loss = 0
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            clean_embeds = text_to_embedding(batch_texts)

            # Sample random timesteps for each example in batch
            timesteps = torch.randint(0, 1000, (len(batch_texts),))

            # Add noise to embeddings
            noisy_embeds, true_noise = add_noise(clean_embeds, timesteps)

            # Predict noise with denoiser (only pass noisy_embeds)
            predicted_noise = denoiser(noisy_embeds)

            # Compute MSE loss
            loss = F.mse_loss(predicted_noise, true_noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss / (len(texts)/batch_size):.4f}")
    return denoiser

In [7]:
def generate_text(prompt, denoiser,steps=50):
  clean_embed = text_to_embedding([prompt])
  noisy_embed = torch.randn_like(clean_embed)

  for t in reversed(range(steps)):
    timestep = torch.full((1,),t, dtype=torch.long)
    with torch.no_grad():
      pred_noise = denoiser(noisy_embed)
      noisy_embed = noise_scheduler.step(pred_noise, t, noisy_embed).prev_sample

  with torch.no_grad():
    # Pass the embedded inputs to the 'bert' attribute
    #outputs = text_encoder.bert(inputs_embeds=noisy_embed)
    # Access the logits from the 'cls' attribute (MLM head)
    logits = text_encoder(inputs_embeds=noisy_embed).logits
    predicted_ids = logits.argmax(dim=-1)
    generated_text = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)
    return generated_text

In [8]:
sample_texts = [
    "The cat sat on the mat",
    "Diffusion models are powerful",
    "Natural Language Processing is evolving",
    "Machine learning requires large datasets",
    "Attention mechanism revolutionalized NLP"
]

In [9]:
trained_denoised = train_denoiser(sample_texts, epochs=5)

Epoch 1/5 - Loss: 1.5040
Epoch 2/5 - Loss: 1.5002
Epoch 3/5 - Loss: 1.5218
Epoch 4/5 - Loss: 1.5636
Epoch 5/5 - Loss: 1.5816


In [10]:
output = generate_text("The future of AI is", trained_denoised)
output

['as as of as and. out']